# 01 — Data Pipeline Debug (config-driven, real TFDS)

Walks the **whole training data pipeline** stage by stage on the *real* TFDS data
named in an experiment config, and visualises boxes + polygons (and distance) at
each step. It reuses `build_input_reader_from_config` — the exact builder training
uses — so it can never drift from the live pipeline.

Stages:
1. Raw decoded detection (TFDS → `PolygonDecoder`)
2. Copy-Paste (source + before/after)
3. Mosaic — 2× canvas + `random_perspective` (rotation/scale/shear/translate, clip-to-edge)
4. Final parser target — PolyYOLO radial round-trip (decode the `[N,72]` target back to vertices)
5. ServingBot distance stream (boxes + metric distance)
6. Merged training batch (detection 128 + distance 16)

**Requirements:** `tfds_data_dir` in the config must contain the datasets
(`cleaner_polygon2026`, `field_misrecog2026`, `station_misrecog`, `cleaner_copy_paste`,
`servingbot_polygon`). Each stage is wrapped so a missing dataset prints guidance
instead of a raw traceback. Edit `CONFIG_PATH` / `N_SHOW` below.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as patches

tf.config.run_functions_eagerly(True)   # so we can .take() and inspect tensors

# ---- knobs ----
CONFIG_PATH = '../configs/experiments/yolo/yolov8_poly_dist.yaml'
N_SHOW      = 4      # samples to display per stage
SEED        = 0
tf.random.set_seed(SEED); np.random.seed(SEED)

from configs.yaml_loader import load_config
from configs.class_map import DETECTION_CLASSES

cfg = load_config(CONFIG_PATH)
data_cfg = cfg.task.train_data
print('config       :', CONFIG_PATH)
print('tfds_name    :', data_cfg.tfds_name)
print('tfds_data_dir:', data_cfg.tfds_data_dir)
print('copy-paste   :', data_cfg.tfds_for_cnp, '(prob', data_cfg.prob_copy_n_paste, ')')
print('distance     :', data_cfg.distance_data.tfds_name if data_cfg.distance_data else None)
print('output size  :', cfg.task.model.input_size[:2], '| classes:', cfg.task.num_classes)

## Visualisation helpers

In [ ]:
CLASS_NAMES = DETECTION_CLASSES   # {id: name}; dict lookup works like a list here

def _u8(img):
    a = np.asarray(img)
    if a.dtype != np.uint8:
        a = np.clip(a * (255.0 if a.max() <= 1.0 + 1e-6 else 1.0), 0, 255).astype(np.uint8)
    return a

def _cname(c):
    return CLASS_NAMES.get(int(c), f'c{int(c)}') if hasattr(CLASS_NAMES, 'get') else str(int(c))

_COLORS = plt.cm.tab20(np.linspace(0, 1, 20))

def _draw_box(ax, box_yxyx, w, h, color, label=None):
    y0, x0, y1, x1 = box_yxyx
    ax.add_patch(patches.Rectangle((x0*w, y0*h), (x1-x0)*w, (y1-y0)*h,
                                   lw=1.5, edgecolor=color, facecolor='none'))
    if label:
        ax.text(x0*w, max(y0*h - 2, 6), label, color='white', fontsize=7,
                bbox=dict(facecolor=color, edgecolor='none', pad=0.5, alpha=0.8))

def _ax_decoded(ax, img, boxes, polys_flat=None, classes=None, draw_poly=True):
    """Decoded/mosaic format: boxes yxyx-norm, polygons flat xy-norm (-1 padded)."""
    u = _u8(img); h, w = u.shape[:2]; ax.imshow(u); ax.axis('off')
    boxes = np.asarray(boxes)
    for i, b in enumerate(boxes):
        col = _COLORS[i % 20]
        lbl = _cname(classes[i]) if classes is not None and i < len(classes) else None
        _draw_box(ax, b, w, h, col, lbl)
        if draw_poly and polys_flat is not None and i < len(polys_flat):
            pts = np.asarray(polys_flat[i]).reshape(-1, 2)            # (x, y) norm
            v = pts[(pts[:, 0] >= 0) & (pts[:, 1] >= 0)]
            if len(v) >= 2:
                xy = np.vstack([v, v[:1]])                            # close ring
                ax.plot(xy[:, 0]*w, xy[:, 1]*h, '-', color=col, lw=1.0, alpha=0.9)

_ANG_STEP = 2*np.pi/24
def _ax_radial(ax, img, bbox, polys72, classes, n_gt, draw_poly=True):
    """Parser target: bbox yxyx-norm, polygons [N,72]=[dist,angle,conf]x24 radial."""
    u = _u8(img); h, w = u.shape[:2]; ax.imshow(u); ax.axis('off')
    bbox = np.asarray(bbox); polys72 = np.asarray(polys72)
    for i in range(int(n_gt)):
        col = _COLORS[i % 20]; b = bbox[i]
        _draw_box(ax, b, w, h, col, _cname(classes[i]))
        if not draw_poly: continue
        dist, ang, conf = polys72[i, 0::3], polys72[i, 1::3], polys72[i, 2::3]
        cy = (b[0]+b[2])/2; cx = (b[1]+b[3])/2
        pts = []
        for k in range(24):
            if conf[k] < 0.5: continue
            theta = (k + np.clip(ang[k], 0, 1)) * _ANG_STEP            # sub-bin offset
            pts.append([(cx + dist[k]*np.cos(theta))*w, (cy + dist[k]*np.sin(theta))*h])
        if len(pts) >= 2:
            xy = np.array(pts + pts[:1])
            ax.plot(xy[:, 0], xy[:, 1], '-o', color=col, lw=1.0, ms=2, alpha=0.9)

def show_row(draw_calls, title):
    """draw_calls: list of callables taking an Axes. Renders them in one row."""
    n = len(draw_calls)
    fig, axes = plt.subplots(1, n, figsize=(4.2*n, 4.2))
    if n == 1: axes = [axes]
    for ax, fn in zip(axes, draw_calls): fn(ax)
    fig.suptitle(title, fontsize=12); fig.tight_layout(); plt.show()

def safe(stage):
    """Decorator: run a stage cell, print guidance on dataset/load errors."""
    def wrap(fn):
        try:
            fn()
        except Exception as e:
            print(f'[{stage}] skipped — {type(e).__name__}: {e}')
            print('  → check that tfds_data_dir contains the dataset and TFDS can read it.')
    return wrap
print('helpers ready')

## Build the pipeline (same components training uses)

In [ ]:
from data_pipeline.input_reader import build_input_reader_from_config

reader = build_input_reader_from_config(
    data_cfg=cfg.task.train_data, task_cfg=cfg.task, is_training=True,
)
decoder   = reader._decoder
cp_module = reader._copy_paste_module
mosaic    = reader._mosaic_module
parser    = reader._parser
dist_rdr  = reader._distance_reader
H, W = cfg.task.model.input_size[:2]

def take(ds, n=N_SHOW):
    return list(ds.take(n))

print('decoder   :', type(decoder).__name__)
print('copy_paste:', type(cp_module).__name__ if cp_module else None)
print('mosaic    :', type(mosaic).__name__ if mosaic else None)
print('parser    :', type(parser).__name__)
print('distance  :', type(dist_rdr).__name__ if dist_rdr else None)

## Stage 1 — Raw decoded detection
TFDS record → `PolygonDecoder.decode`. Boxes (yxyx-norm) + raw polygon vertices.

In [ ]:
@safe('stage1-decoded')
def _():
    raw = reader._load_tfds_datasets()[0]
    dec = raw.map(decoder.decode)
    samples = take(dec)
    calls = []
    for s in samples:
        calls.append(lambda ax, s=s: _ax_decoded(
            ax, s['image'], s['groundtruth_boxes'],
            s.get('groundtruth_polygons'), s.get('groundtruth_classes')))
    show_row(calls, 'Stage 1 — raw decoded (boxes + polygon vertices)')

## Stage 2 — Copy-Paste
Detection sample zipped with the copy-paste source (RGBA), pasted before mosaic.
Set `FORCE_COPY_PASTE=True` to paste on every sample (the config probability is < 1, so most samples are otherwise unchanged).

In [ ]:
FORCE_COPY_PASTE = True

@safe('stage2-copypaste')
def _():
    if cp_module is None:
        print('copy-paste not configured for this run.'); return
    if FORCE_COPY_PASTE and hasattr(cp_module, '_prob'):
        cp_module._prob = 1.0
    raw = reader._load_tfds_datasets()[0]
    dec = raw.map(decoder.decode)
    cnp = reader._load_cnp_dataset()
    # copy-paste source preview (RGB + alpha if present)
    try:
        src = take(cnp, 1)[0]
        si = _u8(src['image'])
        fig, axs = plt.subplots(1, 2, figsize=(8, 4))
        axs[0].imshow(si[..., :3]); axs[0].set_title('copy-paste source (RGB)'); axs[0].axis('off')
        if si.shape[-1] == 4:
            axs[1].imshow(si[..., 3], cmap='gray'); axs[1].set_title('alpha mask')
        axs[1].axis('off'); plt.tight_layout(); plt.show()
    except Exception as e:
        print('source preview skipped:', e)
    # before / after copy-paste on the same detection sample
    after = tf.data.Dataset.zip((dec, cnp)).map(cp_module.process_fn(is_training=True))
    befores, afters = take(dec), take(after)
    calls = []
    for b in befores[:N_SHOW//2 or 1]:
        calls.append(lambda ax, b=b: (_ax_decoded(ax, b['image'], b['groundtruth_boxes'],
                     b.get('groundtruth_polygons'), b.get('groundtruth_classes')), ax.set_title('before'))[0])
    for a in afters[:N_SHOW//2 or 1]:
        calls.append(lambda ax, a=a: (_ax_decoded(ax, a['image'], a['groundtruth_boxes'],
                     a.get('groundtruth_polygons'), a.get('groundtruth_classes')), ax.set_title('after'))[0])
    show_row(calls, 'Stage 2 — copy-paste (before vs after)')

## Stage 3 — Mosaic (2× canvas + random_perspective)
Pre-resize → `padded_batch(group_size)` → `mosaic_fn` (G in → G//R out) → unbatch. Set `FORCE_MOSAIC=True`
to take the mosaic branch every time (the config frequency is < 1).

In [ ]:
FORCE_MOSAIC = True

@safe('stage3-mosaic')
def _():
    if mosaic is None:
        print('mosaic not configured.'); return
    if FORCE_MOSAIC:
        mosaic._mosaic_freq = 1.0
    raw = reader._load_tfds_datasets()[0]
    dec = raw.map(decoder.decode)
    def _pre(ex, H=H, W=W):
        img = tf.cast(tf.image.resize(tf.cast(ex['image'], tf.float32), [H, W], 'bilinear'), tf.uint8)
        return {**ex, 'image': img}
    mos = (dec.map(_pre).padded_batch(mosaic._group_size, drop_remainder=True)
              .map(mosaic.mosaic_fn(is_training=True)).unbatch())
    samples = take(mos)
    calls = []
    for s in samples:
        calls.append(lambda ax, s=s: _ax_decoded(
            ax, s['image'], s['groundtruth_boxes'],
            s.get('groundtruth_polygons'), s.get('groundtruth_classes')))
    show_row(calls, 'Stage 3 — mosaic + random_perspective')

## Stage 4 — Final parser target (PolyYOLO radial round-trip)
Full `parse_fn(is_training=True)`. We decode the `[N,72]` radial target **back** to
vertices using `θ = (k + sub-bin offset)·15°` — if the overlay hugs the object,
the radial encoding is faithful. Labels are class-named.

In [ ]:
@safe('stage4-parser')
def _():
    if FORCE_MOSAIC and mosaic is not None:
        mosaic._mosaic_freq = 1.0
    raw = reader._load_tfds_datasets()[0]
    dec = raw.map(decoder.decode)
    def _pre(ex, H=H, W=W):
        img = tf.cast(tf.image.resize(tf.cast(ex['image'], tf.float32), [H, W], 'bilinear'), tf.uint8)
        return {**ex, 'image': img}
    ds = dec.map(_pre).padded_batch(mosaic._group_size, drop_remainder=True).map(mosaic.mosaic_fn(is_training=True)).unbatch()
    ds = ds.map(parser.parse_fn(is_training=True))
    samples = take(ds)
    calls = []
    for img, lab in samples:
        calls.append(lambda ax, img=img, lab=lab: _ax_radial(
            ax, img, lab['bbox'], lab['polygons'], lab['classes'], lab['n_gt']))
    show_row(calls, 'Stage 4 — parser target (radial polygon round-trip)')

## Stage 5 — ServingBot distance stream
Separate, training-only stream: decode → `V8DistanceParser`. Boxes are annotated
with the metric distance `exp(log_distance)` (sentinel `≤ −10` = no distance).

In [ ]:
@safe('stage5-distance')
def _():
    if dist_rdr is None:
        print('distance stream not configured (with_distance=False).'); return
    raw = dist_rdr._load_tfds_datasets()[0]
    ds = raw.map(dist_rdr._decoder.decode).map(dist_rdr._parser.parse_fn(is_training=True))
    samples = take(ds)
    calls = []
    for img, lab in samples:
        def draw(ax, img=img, lab=lab):
            u = _u8(img); h, w = u.shape[:2]; ax.imshow(u); ax.axis('off')
            n = int(lab['n_gt']); bx = np.asarray(lab['bbox']); ld = np.asarray(lab['log_distance'])
            cl = np.asarray(lab['classes'])
            for i in range(n):
                col = _COLORS[i % 20]
                m = float(np.exp(ld[i])) if ld[i] > -10 else None
                lbl = f'{_cname(cl[i])}' + (f' {m:.2f}m' if m is not None else ' (no-d)')
                _draw_box(ax, bx[i], w, h, col, lbl)
        calls.append(draw)
    show_row(calls, 'Stage 5 — distance stream (boxes + metric distance)')

## Stage 6 — Merged training batch
The full `reader()` output: detection (128) + distance (16) concatenated on the
batch axis. We show a few of each; `ignore_bg=1` marks the distance-only samples.

In [ ]:
@safe('stage6-merged')
def _():
    batch = take(reader(), 1)[0]
    imgs, lab = batch
    imgs = imgs.numpy(); ig = np.asarray(lab['ignore_bg']); B = imgs.shape[0]
    det_idx = [i for i in range(B) if ig[i] == 0][:N_SHOW//2 or 1]
    dst_idx = [i for i in range(B) if ig[i] == 1][:N_SHOW//2 or 1]
    print(f'batch={B}  detection(ignore_bg=0)={int((ig==0).sum())}  distance(ignore_bg=1)={int((ig==1).sum())}')
    calls = []
    for i in det_idx:
        calls.append(lambda ax, i=i: (_ax_radial(ax, imgs[i], lab['bbox'][i].numpy(),
                     lab['polygons'][i].numpy(), lab['classes'][i].numpy(), int(lab['n_gt'][i])),
                     ax.set_title(f'det #{i}'))[0])
    for i in dst_idx:
        calls.append(lambda ax, i=i: (_ax_radial(ax, imgs[i], lab['bbox'][i].numpy(),
                     lab['polygons'][i].numpy(), lab['classes'][i].numpy(), int(lab['n_gt'][i])),
                     ax.set_title(f'dist #{i}'))[0])
    show_row(calls, 'Stage 6 — merged batch (detection + distance)')

---
**Tips**
- Stage 4 is the key correctness check: the radial overlay should track the Stage 1/3 polygon. If it lags by a constant angle, suspect the sub-bin offset decode `(k+offset)·15°`.
- Toggle `FORCE_MOSAIC` / `FORCE_COPY_PASTE` off to see the natural (config-frequency) mix.
- To debug a *different* experiment, just change `CONFIG_PATH` — everything else is config-driven.
- The geometric augmentation lives in the mosaic stage (`random_perspective`), applied to mosaic **and** single-image branches; the parser no longer does a separate affine.